# 常駐 Spark クラスタで実行する

`compute` が有効なら Master / Worker へ接続します。無効なら `local[2]` と明示します。接続失敗時の自動 local 切替はありません。最初に実行モードを確認してください。

In [ ]:
from lakehouse_session import execution_settings, session_info, spark_session

execution_settings()

In [ ]:
spark = spark_session()
# 実際の Master、application ID、Master / Worker / History のリンク。
session_info(spark)

## Python executor の実行場所と依存を確認する

remote の結果は Worker container、local の結果は Jupyter container の hostname です。これは driver の情報を表示しているだけではなく、2 partition の Python task が返した結果です。

In [ ]:
def executor_details(partition):
    import platform
    import socket

    import numpy
    import pandas
    import pyarrow

    yield {
        "hostname": socket.gethostname(),
        "python": platform.python_version(),
        "pandas": pandas.__version__,
        "pyarrow": pyarrow.__version__,
        "numpy": numpy.__version__,
    }


executor_rows = (
    spark.sparkContext.parallelize(range(4), 2)
    .mapPartitions(executor_details)
    .collect()
)
executor_rows

In [ ]:
def label_batches(batches):
    for batch in batches:
        batch["label"] = batch["id"].map(lambda value: f"row-{value}")
        yield batch


# Pandas + Arrow の処理も executor 側で実行します。
spark.range(4).mapInPandas(
    label_batches, schema="id LONG, label STRING"
).show()

## Iceberg の小さな round trip

通常の Bronze / Silver / Gold へは書きません。`notebook_demo.cluster_roundtrip` のみに合成データを保存します。

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.notebook_demo")
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.notebook_demo.cluster_roundtrip (
        id BIGINT, label STRING
    ) USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")
spark.createDataFrame(
    [(1, "synthetic cluster notebook")],
    schema="id LONG, label STRING",
).createOrReplaceTempView("notebook_cluster_input")
spark.sql("""
    MERGE INTO lakehouse.notebook_demo.cluster_roundtrip AS target
    USING notebook_cluster_input AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("""
    SELECT id, label
    FROM lakehouse.notebook_demo.cluster_roundtrip
    ORDER BY id
""").show(truncate=False)

## 実行終了と履歴

実行中は Master の application リンクから job / stage / executor を確認します。`spark.stop()` の後、History UI の一覧に application が現れます。反映には数秒かかることがあります。Notebook を保存した後は kernel も shutdown してください。

In [ ]:
application_id = spark.sparkContext.applicationId
spark.stop()
print("Completed application:", application_id)